#  YOLOv5 学习总结

> 本周完整走通了 YOLOv5 的**理论 → 源码 → 训练 → 评估 → 推理部署**全流程，并在 PCB 缺陷数据集上完成了实战训练。

---

## 一、学习路线总览

```mermaid
graph TD
    A["📅 Day1 (0727)<br/>项目结构 + 模型架构"] --> B["📅 Day2 (0728)<br/>核心机制: 损失/数据/训练"]
    B --> C["📅 Day3 (0729)<br/>PCB 实战训练 + 评估"]
    C --> D["📅 Day4 (0730-0731)<br/>detect.py 推理部署"]
    D --> E["🎯 最终产出<br/>best.pt + mAP@0.5=92.1%"]
```

| 日期 | 主题 | 关键知识点 |
|---|---|---|
| **0727** | 项目结构 & 模型架构 | 顶层脚本、`models/`、`utils/`、`yolo.py`、C3、SPPF、v3→v5 改进 |
| **0728** | 训练核心机制 | `parse_model`、损失函数、数据加载、训练循环 |
| **0729** | PCB 实战 + 评估 | PCB 6 类缺陷训练、P/R/mAP 指标、结果可视化 |
| **0731** | 推理部署 | `detect.py` 推理、参数解析、结果保存 |

---

## 二、Day1 — 模型架构（0727）

### 2.1 项目结构（顶层 → 核心 → 工具）

| 层次 | 内容 | 作用 |
|---|---|---|
| **顶层脚本** | `detect.py` / `train.py` / `val.py` / `export.py` | 推理 / 训练 / 验证 / 导出四大入口 |
| **models/** | `yolo.py` + `common.py` + 5 个 `.yaml` | 配置驱动的模型构建引擎 |
| **utils/** | `dataloaders` / `loss` / `metrics` / `autoanchor` | 数据、损失、评估、锚框等工具库 |
| **data/** | `coco128.yaml` / `hyps/` | 数据集配置 + 超参数 |

### 2.2 YOLOv5 vs YOLOv3 六大改进

| 维度 | YOLOv3 | YOLOv5 | 收益 |
|---|---|---|---|
| 骨干 | Darknet-53 | **CSPDarknet53** | 计算量 -20% |
| Neck | 仅 FPN | **FPN + PAN** | 定位+语义双向融合 |
| 激活 | LeakyReLU | **SiLU** | +0.5~1% mAP |
| 正样本 | 1 anchor × 1 grid | **3 anchor × 3 grid** | 收敛更快、小目标召回↑ |
| 回归损失 | IoU | **CIoU** | 框得更准 |
| 增强 | 标准 | **Mosaic + MixUp + Multi-Scale** | 泛化能力↑ |

> 🎯 YOLOv5s 用 **1/9 的参数**、**1/9 的计算量**，mAP 反超 YOLOv3 4.4 个百分点。

### 2.3 两大核心模块

**C3（CSP Bottleneck with 3 Convolutions）** — "双通道快递分拣站"
```python
def forward(self, x):
    return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))
```
- 一条路：`cv1 → Bottleneck×n`（深层特征提取）
- 另一条路：`cv2`（保留原始信息）
- 最后 `cat` 拼接 + `cv3` 融合

**SPPF（快速空间金字塔池化）** — 一个池化串行复用 3 次
- 1 个 `MaxPool(k=5)` 串行 3 次 → 等效感受野 **5 / 9 / 13**
- 相比 SPP 并行 3 个池化层**快 2~3 倍**，效果不变

---

## 三、Day2 — 训练核心机制（0728）

### 3.1 `parse_model()`：从 YAML 到模型

```
yolov5s.yaml ──parse_model──▶ nn.Sequential 模型
  gd=0.33 (深度系数)  →  C3 中 Bottleneck 重复次数
  gw=0.50 (宽度系数)  →  每层卷积输出通道数
```

> 改 `gd`/`gw` 两个参数即可得到 n/s/m/l/x 五种模型，**不改一行代码**。

### 3.2 损失函数 `ComputeLoss`

$$\text{Loss} = \lambda_{box} \cdot \text{CIoU} + \lambda_{obj} \cdot \text{BCE} + \lambda_{cls} \cdot \text{BCE}$$

| 损失 | 权重 | 作用 | 只看哪些位置 |
|---|---|---|---|
| **L_box** | 0.05 | 框得准不准（CIoU） | 仅正样本 |
| **L_obj** | 1.0 | 这里有东西吗（BCE，target=IoU） | 全部位置 |
| **L_cls** | 0.5 | 是啥类别（BCE + 标签平滑） | 仅正样本 |

**正样本匹配（`build_targets`）**：每个 GT 匹配最多 **3 anchor × 3 相邻 grid = 9 个正样本**，这是加速收敛的关键。

### 3.3 数据加载 `dataloaders.py`

```mermaid
graph LR
    A["扫描图片+标签"] --> B["缓存 .cache"]
    B --> C["Mosaic 4图拼接"]
    C --> D["MixUp / HSV / 翻转 / 缩放"]
    D --> E["输出 [3,640,640]"]
```

- **Mosaic**：4 图拼 1 张 → 变相 batch×4，去掉后 mAP 降 3~5%
- **Rect 矩形训练**：同 batch 按宽高比分组 → 少 padding → 提速 30%

### 3.4 训练循环 `train.py`

| 技巧 | 作用 |
|---|---|
| **Warmup**（前 3 轮） | lr 从 0 缓慢升温，防止起步震荡 |
| **Multi-Scale** | 每批随机 320~960 尺寸，尺度鲁棒 |
| **梯度累积** | 小 batch 模拟大 batch（nbs=64） |
| **AMP 混合精度** | 速度 ×2，显存 ÷2 |
| **EMA** | 参数平滑，验证更准 |
| **EarlyStopping** | 连续 patience 轮不提升即停 |

---

## 四、Day3 — PCB 实战训练 + 评估（0729）

### 4.1 训练配置与结果

| 配置 | 值 | 指标 | 值 |
|---|---|---|---|
| 模型 | YOLOv5s（预训练） | **mAP@0.5** | **92.1%** |
| 数据集 | PCB 6 类缺陷 | **mAP@0.5:0.95** | **63.5%** |
| 尺寸 / Batch | 640 / 16 | **Precision** | **98.4%** |
| Epochs | 100（~3.1h, RTX 3060） | **Recall** | **90.6%** |

### 4.2 六类缺陷表现

| 类别 | mAP@0.5 | 评估 |
|---|---|---|
| `open_circuit` 断路 | 95.9% | 🟢 优秀 |
| `spur` 毛刺 | 94.1% | 🟢 优秀 |
| `missing_hole` 缺孔 | 93.2% | 🟢 优秀 |
| `short` 短路 | 92.5% | 🟢 优秀 |
| `spurious_copper` 多余铜 | 88.9% | 🟡 良好 |
| `mouse_bite` 缺口 | 88.2% | 🟡 良好 |

> 推理速度 **~12ms/张（83 FPS）**，达实时检测水平，可部署产线。

### 4.3 评估指标速查

| 指标 | 含义 | 我们的值 |
|---|---|---|
| **P** 精确率 | 检出的有多少是对的 | 98.4% ✅ 误检极少 |
| **R** 召回率 | 真实缺陷找出多少 | 90.6% ✅ 漏检较低 |
| **mAP@0.5** | 宽松标准下检测准不准 | 92.1% ✅ 优秀 |
| **mAP@0.5:0.95** | 严格标准下框得准不准 | 63.5% 🟡 可优化 |

**训练产物**：`runs/train/exp4/weights/best.pt` + `results.png` / `PR_curve.png` / `confusion_matrix.png` / `labels.jpg` / `F1_curve.png`

---

## 五、Day4 — 推理部署（detect.py）

```mermaid
graph LR
    A["--weights best.pt"] --> B["DetectMultiBackend<br/>加载模型"]
    B --> C["LoadImages<br/>数据加载"]
    C --> D["前向推理 + NMS"]
    D --> E["scale_boxes 缩放回原图"]
    E --> F["标注 + 保存 runs/detect/"]
```

```bash
python detect.py --weights runs/train/exp4/weights/best.pt \
                 --source PCBYOLODataset/images/val \
                 --data PCBYOLODataset/pcb.yaml \
                 --conf-thres 0.5 --save-txt
```

**关键参数**：`--conf-thres`（置信度阈值，调高 → P↑ R↓）、`--iou-thres`（NMS 阈值）、`--save-txt`（保存坐标结果）、`--device`（0=cuda / cpu）

---

## 六、本周核心收获一句话

> **一条主线**：`yaml 配置 → parse_model 建模型 → Mosaic 数据加载 → ComputeLoss 三损失 → train 训练循环（Warmup/AMP/EMA）→ val 评估（P/R/mAP）→ detect 推理部署`，最终在 PCB 缺陷数据集上达到 **mAP@0.5=92.1%、P=98.4%、83 FPS** 的工业可用水平。

## 七、后续可优化方向

1. 🔄 增大模型（`yolov5m/l`）或增加 epochs（150~200）提升 mAP
2. 🔄 针对 `mouse_bite` / `spurious_copper` 补充数据 + 增强
3. 🔄 `export.py` 导出 ONNX / TensorRT 加速部署
4. 🔄 调 `conf-thres` 在精确率/召回率间按需求权衡


# 八、YOLOv5 模型评估指标详解

> 训练结束后 `val.py` 会在验证集上计算以下指标，并输出到 `results.png` / `PR_curve.png` 等文件中，用于衡量模型的**检测准确度**与**定位精度**。

## 8.1 基础概念

### IoU（交并比，Intersection over Union）

$$\text{IoU} = \frac{\text{预测框} \cap \text{真实框}}{\text{预测框} \cup \text{真实框}}$$

- 衡量**预测框与真实框的重合程度**，取值 0~1，越大框得越准。
- 它是判定"检测对没对"的**唯一标准**：IoU ≥ 阈值才算匹配成功（默认 0.5）。
- 同时也是 NMS 去重的依据。

### TP / FP / FN

| 符号 | 名称 | 含义 | 通俗理解 |
|---|---|---|---|
| **TP** | 真正例 | 预测有缺陷，且 IoU ≥ 阈值，确实有缺陷 | 检对了 |
| **FP** | 假正例 | 预测有缺陷，但实际没有（或 IoU < 阈值） | 误检 |
| **FN** | 假负例 | 实际有缺陷，但模型没检测出来 | 漏检 |

> 所有指标都是由这三个数字推导出来的，**先有 TP/FP/FN，才有 P/R/mAP**。

## 8.2 核心指标

### Precision（精确率 / 准确率）P

$$P = \frac{TP}{TP + FP} = \frac{\text{检出的正确框数}}{\text{全部检出框数}}$$

- **检出的框中有多少是对的** → 衡量"误检"（FP）多不多。
- P = 98.4% 表示：检出 100 个框，约 98 个是真正的缺陷，误检极少。

### Recall（召回率）R

$$R = \frac{TP}{TP + FN} = \frac{\text{检出的正确框数}}{\text{真实缺陷总数}}$$

- **真实缺陷中被找出来的比例** → 衡量"漏检"（FN）多不多。
- R = 90.6% 表示：100 个真实缺陷中能找到约 91 个，漏检较低。

### Precision 与 Recall 的权衡（conf-thres）

- **提高** `--conf-thres` → 只保留高置信度框 → P ↑ 但 R ↓（宁可漏检也不误检）
- **降低** `--conf-thres` → 保留更多框 → R ↑ 但 P ↓（宁可误检也不漏检）
- 单一指标无法代表模型好坏，所以引入 **PR 曲线** 与 **AP**。

### AP（平均精度，Average Precision）

- 以 Recall 为横轴、Precision 为纵轴画出 **PR 曲线**，曲线下的面积就是 AP。
- **AP 综合了 P 和 R 的权衡**，是单个类别检测能力的总体度量。
- AP 越高 → 曲线越靠近右上角 → 模型在"又准又全"上表现越好。

### mAP（平均精度均值，mean Average Precision）

$$\text{mAP} = \frac{1}{N}\sum_{i=1}^{N}\text{AP}_i$$

- 把所有类别的 AP 取平均，得到**整个模型的检测能力**。
- YOLOv5 训练日志中通常看两个版本：

| 指标 | 含义 | 特点 | 我们的值 |
|---|---|---|---|
| **mAP@0.5** | IoU 阈值 = 0.5 时的 mAP | 宽松标准，主要看"检没检出" | **92.1%** ✅ |
| **mAP@0.5:0.95** | IoU 从 0.5 到 0.95 按 0.05 步长取 10 档 mAP 的平均 | 严格标准，同时要求"框得准"，更能反映定位精度 | **63.5%** 🟡 |

> 📌 **mAP@0.5 高而 mAP@0.5:0.95 低** → 说明"能检出目标但框不够精确"，是后续优化的重点方向（如调回归损失权重、增大模型）。

## 8.3 辅助评估图

| 图 | 作用 |
|---|---|
| **PR_curve.png** | 每个类别的 P-R 关系曲线，曲线下面积 = AP，越靠右上越好 |
| **F1_curve.png** | 不同置信度下的 F1 值，F1 = 2PR/(P+R)，P 与 R 的调和平均 |
| **confusion_matrix.png** | 混淆矩阵，对角线越亮越好，可看出误检/漏检集中在哪两类 |
| **labels.jpg** | 标注可视化，检查数据集的真实分布是否合理 |
| **results.png** | 训练过程曲线（loss / P / R / mAP），看收敛情况 |

## 8.4 一句话总结

> **P 管"误检"、R 管"漏检"、mAP@0.5 管"检没检出"、mAP@0.5:0.95 管"框得准不准"**，四个指标结合 PR 曲线 / 混淆矩阵，才能全面评价一个 YOLOv5 模型的好坏。